# Agent란?
- LLM이 스스로 판단해서 어떤 행동(툴 사용 포함)을 할 지 결정하는 실행주체를 의미합니다.

In [1]:
from dotenv import load_dotenv
from langchain_openai.chat_models.base import ChatOpenAI
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-VZytJ5BN3ErM


## agent를 만들어야 하는이유

In [3]:
# 5,243.38
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

In [4]:
chat = ChatOpenAI(temperature=0.1)

In [5]:
result = chat.invoke(prompt)

In [6]:
# 5,243.38
result.content

'The total cost is $4,363.38.'

In [6]:
chat = ChatOpenAI(model="gpt-4o" ,temperature=0.1)

In [7]:
# 5243.38
result = chat.invoke(prompt)

In [8]:
# 5,243.38
print(result.content)

To find the total cost, you need to add all the amounts together:

\[ 
355.39 + 924.87 + 721.2 + 1940.29 + 573.63 + 65.72 + 35.00 + 522.00 + 76.16 + 29.12 = 5243.38 
\]

So, the total cost is $5243.38.


In [ ]:
"""
    프롬프트의 정답
    $4,363.38.

    계산기에서 직접 계산하기 ↓↓↓
    $5,243.38

    llm의 계산 착오
    ※이유※
    LLM은 산술 연산을 수행하지 않습니다. 이런 계산은 AI보다 계산기가 더 잘합니다.
    LLM은 text를 생성해내는 모델입니다. 문장의 시퀀스의 다음 token이 무엇인지 통계적으로 추측합니다.
    이러한 LLm의 오류를 잡기 위해서는 agent를 제공해주어야 합니다.
    그리고 agent를 위한 tool(툴)을 만들고, agent가 tool을 선택해서 실행하는 것입니다.
"""

## Agent 생성

In [2]:
# create_agent: 함수로 통일
from langchain.agents import create_agent
from langchain.tools import tool

In [18]:
@tool
def plus(num1: float, num2: float) -> float:
    """
        Adds two numbers and return the result.
    """
    return num1 + num2


"""
    json {
        "name": plus,
        "description": "Adds two numbers and return the result.",
        "parameters": {
            "num1": "float",
            "num2": "float",
        }
    
    }

"""

None

In [20]:
agent = create_agent(
    model="gpt-3.5-turbo",
    tools=[plus],
    system_prompt="You are a helpful assistant"
)

In [21]:
result = agent.invoke({
    "messages": [
        {
            "role": "user", 
            "content": prompt
        }
    ]
})

In [22]:
# 5243.38
result

{'messages': [HumanMessage(content='cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12', additional_kwargs={}, response_metadata={}, id='66308184-6de3-4a5f-878c-83b765e55e5b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 109, 'total_tokens': 236, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dh4NK9xfz60dywtjQ4yGLcjbIEQOJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3e09-d644-7142-8c1e-4c73e5bb8397-0', tool_calls=[{'name': 'plus', 'args': {'num1': 355.39, 'num2': 924.87}, 'id': 'call_kxfLZcVPA7gqb7Oj4mbN3Y11', 'type': 'tool_call'},

In [29]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'plus', 'args': {'num1': 355.39, 'num2': 924.87}, 'id': 'call_kxfLZcVPA7gqb7Oj4mbN3Y11', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 721.2, 'num2': 1940.29}, 'id': 'call_b7TOJlvvZkvQnGq0Ija3n4JW', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 573.63, 'num2': 65.72}, 'id': 'call_9xeBodBE9Qkn6rz1Vh3JlRDy', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 35, 'num2': 522}, 'id': 'call_dZhGgw3IWt9A9el9Cvzza7yn', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 76.16, 'num2': 29.12}, 'id': 'call_ePc1ioCptnveC2ts5MFWsIho', 'type': 'tool_call'}


In [3]:
@tool
def total_sum(numbers: list[float]) -> float:
    """
        Adds a list of numbers and returns the total sum.
        Use this tool when you need to calculate the total of multiple numbers.
        Input should be a string representation of a list.
        Example: "[1, 2, 3]"
    """
    return sum(numbers)

In [4]:
agent = create_agent(
    model="gpt-3.5-turbo",
    tools=[total_sum],
    system_prompt="You are a helpful assistant"
)

In [6]:
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

result = agent.invoke({
    "messages": [
        {
            "role": "user", 
            "content": prompt
        }
    ]
})

In [36]:
# $5243.38
result["messages"][-1].content

'The total cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12 is $5243.38.'

In [37]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'total_sum', 'args': {'numbers': [355.39, 924.87, 721.2, 1940.29, 573.63, 65.72, 35, 522, 76.16, 29.12]}, 'id': 'call_vUYuuXgj9LCproxGdBhrYzav', 'type': 'tool_call'}


# LangSmith(랭스미스)
- LLM 기반 애플리케이션의 디버깅, 성능 평가, 모니터링 등을 제공하는 랭체인의 통합 플랫폼입니다.

## 랭스미스 Open API Key 발급
- https://smith.langchain.com/ 접속
- 로그인 후 좌측 하단 [Setting] 메뉴 클릭
- [API Keys] 클릭 후 생성
- Description은 lang_ksh(이니셜)
- [default workspace]는 기존에 있는 workspace1로 만들고 생성
- 발급받은 KEY를 .env에 추가하기

### .env에 추가하기
- LANGCHAIN_TRACING_V2=true
- LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"
- LANGCHAIN_PROJECT=lang_1900
- LANGSMITH_API_KEY=발급받은 랭스미스 key

### 설정 후 Jupyter Notebook 재실행

## Agent가 동작하는 과정
1. 끝날때까지 반복이 되는 loop입니다.
2. llm으로 부터 어떤 것을 할지(get action)을 받아온다. (lang smith의 output에서 확인가능)
3. 실행한 결과를 observation이라고 부른다. 다시 다음 next action을 실행시킨다.
4. Agent Finish를 응답받으면 마지막 action 값을 리턴한다.

## 1. ReAct (Reasoning and Acting) Agent

In [8]:
from dotenv import load_dotenv
import os

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain_classic.agents import AgentExecutor, create_react_agent, create_openai_functions_agent
from langchain_classic import hub
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List #Python의 내장 모듈 typing

In [9]:
load_dotenv()

True

In [10]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [11]:
@tool
def plus(expression: str) -> float:
    """
        Adds multiple numbers and returns their total sum.

        The input must be a comma-spreated string of numbers.
        Example: "10,20,30"
        
        Use this tool when you nee to calcuate the sum of multiple values.
    """
    try:
        numbers = [float(num) for num in expression.split(",")]
        return sum(numbers)
    except Exception as e:
        return -1

In [15]:
tools = [plus]
react_agent_prompt = hub.pull("hwchase17/react")

# 판단
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt = react_agent_prompt
)

# 실행기
react_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # 내부 동작 확인
)

In [16]:
# 5,760.38
prompt = "cost of $3215.39 + $35.87 + $45.2 + $123.29 + $535.63 + $8.72 + $35.00 + $453.00 + $76.16 + $1232.12"

result = react_agent_executor.invoke({
    "input": prompt
})



> Entering new AgentExecutor chain...
To find the total cost, I need to sum all the given amounts. I will use the plus function to calculate the total.

Action: plus
Action Input: "3215.39,35.87,45.2,123.29,535.63,8.72,35.00,453.00,76.16,1232.12"5760.379999999999I now know the final answer.  
Final Answer: 5760.38

> Finished chain.


## 2. OpenAI Function Calling Agent

In [19]:
class CalculatorToolArgsSchema(BaseModel):
    numbers: List[float] = Field(description="Numbers to sum")

class CalculatorTool(BaseTool):
    # 약속된 필드 이름 (공백x, 한글x, a-z, A-Z, 0-9, _, -만 가능)
    name: Type[str] = "calculator_tool"
    description: Type[str] = """
        Adds multiple numbers and returns their total sum.
        Use this tools when you need to calculator the sum of multiple values.
    """

    args_schema: Type[BaseModel] = CalculatorToolArgsSchema

    # BaseTool은 반드시 _run 함수를 재정의
    # tool을 호출했을 때 실행되는 메인로직
    def _run(self, numbers):
        return sum(numbers)

In [30]:
tools = [CalculatorTool()]

# placeholder(agent_scratchpad): 내부 tool, reasoning 호출 기록을 임시 저장
function_agent_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("placeholder", """{agent_scratchpad}"""),
])

# 판단
agent = create_openai_functions_agent(
    llm=llm,
    tools=tools,
    prompt=function_agent_prompt
)

# 실행기
calling_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [31]:
# 92,988.59
prompt = "cost of $45353.39 + $35.87 + $45353.42 + $1221.29 + $535.63 + $8.72 + $123.00 + $45.00 + $76.56 + $235.71"

result = calling_agent_executor.invoke({
    "input": prompt
})



> Entering new AgentExecutor chain...

Invoking: `calculator_tool` with `{'numbers': [45353.39, 35.87, 45353.42, 1221.29, 535.63, 8.72, 123, 45, 76.56, 235.71]}`


92988.59The total cost is $92,988.59.

> Finished chain.


In [33]:
result["output"]

'The total cost is $92,988.59.'

# v1.0↑ create_agent(커스텀 툴)

In [48]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List, Tuple, Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-VZytJ5BN3ErM


In [3]:
tools = []

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘!")
])

chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 현재 실시간 날씨 알려줘!', additional_kwargs={}, response_metadata={}, id='7ebe228d-888f-4f4c-b7e3-3568f037d805'),
  AIMessage(content='죄송하지만, 실시간 날씨 정보를 제공할 수는 없습니다. 하지만 강남 날씨를 확인하려면 기상청 웹사이트나 날씨 앱을 이용해보시기를 추천드립니다. 현재 날씨와 예보를 쉽게 확인할 수 있습니다. 다른 질문이 있으면 도와드릴 수 있습니다!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 18, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_da1f8e43b9', 'id': 'chatcmpl-Dh6zAMLEDEhwJQuEkNOnWQYVw8g9W', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e3ea2-f356-7f13-90be-499175e520be-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'outp

In [38]:
result["messages"][-1].content

'죄송하지만, 실시간 날씨 정보를 제공할 수는 없습니다. 하지만 강남 날씨를 확인하려면 기상청 웹사이트나 날씨 앱을 이용해보시기를 추천드립니다. 현재 날씨와 예보를 쉽게 확인할 수 있습니다. 다른 질문이 있으면 도와드릴 수 있습니다!'

In [45]:
# 지역 -> 위도, 경도
def get_coordinates(location_name):
    locator = Nominatim(user_agent="ksh")
    location = locator.geocode(location_name)

    return location.latitude, location.longitude, 

In [54]:
get_coordinates("선릉역")

(37.5057908, 127.0483487)

In [36]:
# 위도, 경도 -> 날씨
def get_weather(lat, lon):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    
    # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
    weather_codes = {
        0: "맑음 ☀️",
        1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
        45: "안개 🌫️", 48: "침강 안개 🌫️",
        51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
        61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
        71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
        80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
        95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
    }

    try:
        response = requests.get(url)
        datas = response.json()

        if "current_weather" in datas:
            current = datas["current_weather"]
            temp = current["temperature"]
            wind = current["windspeed"]
            code = current["weathercode"]

            condition = weather_codes.get(code, "알 수 없음")

            return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}km/h"
        
    except Exception as e:
        return "요청 실패"

In [37]:
print(get_weather(37.500078, 127.035548))

상태: 흐림 ☁️
온도: 23.1°C
풍속: 5.4km/h


## 함수 -> 툴로 변경 후 제공

In [64]:
class CoordinatesToolArgSchema(BaseModel):
    location_name: str = Field("위도와 경도로 바꾸고 싶은 장소명입니다.")

class CoordinatesTool(BaseTool):
    name: Type[str] = "coordinates_tool"
    description: Type[str] = """
        장소명을 위도(latitude)와 경도(longitude) 좌표로 변환합니다.
        장소명을 위도와 경도로 변환하고 싶을 대 사용하는 도구입니다.
    """
    args_schema: Type[BaseModel] = CoordinatesToolArgSchema

    def _run(self, location_name: str) -> Tuple[float, float]:
        locator = Nominatim(user_ageqnt="ksh")
        location = locator.geocode(location_name)
    
        return location.latitude, location.longitude, 

In [63]:
class WeatherSearchToolArgSchema(BaseModel):
    lat: float = Field(description="위도, Example Value: 37.500078")
    lon: float = Field(description="경도, Example Value: 127.035548")
    
class WeatherSearchTool(BaseTool):
    name: Type[str] = "weather_search_tool"
    description: Type[str] = """
        지역의 날씨를 가져오고 싶을 때 사용하는 툴입니다.
        위도와 경도를 입력하면, 해당 지역의 날씨의 정보를 문자열로 반환합니다.
    """

    args_schema: Type[BaseModel] = WeatherSearchToolArgSchema
    
    # 위도, 경도 -> 날씨
    def _run(self, lat: float, lon: float) -> str:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        
        # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
        weather_codes = {
            0: "맑음 ☀️",
            1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
            45: "안개 🌫️", 48: "침강 안개 🌫️",
            51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
            61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
            71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
            80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
            95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
        }
    
        try:
            response = requests.get(url)
            datas = response.json()
    
            if "current_weather" in datas:
                current = datas["current_weather"]
                temp = current["temperature"]
                wind = current["windspeed"]
                code = current["weathercode"]
    
                condition = weather_codes.get(code, "알 수 없음")
    
                return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}km/h"
            
        except Exception as e:
            return "요청 실패"

In [66]:
tools = [CoordinatesTool(), WeatherSearchTool()]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘!")
])

chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 현재 실시간 날씨 알려줘!', additional_kwargs={}, response_metadata={}, id='ccfead8f-decd-4dc9-86e1-d01ce8f4e1b9'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 185, 'total_tokens': 201, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_22a0c4db5d', 'id': 'chatcmpl-Dh7pzVoSjI0AJT6vkwqpegWem5r7f', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3ed4-ede3-7f40-96b4-db087406974d-0', tool_calls=[{'name': 'coordinates_tool', 'args': {'location_name': '강남'}, 'id': 'call_UvEM7W7HjRS6lxQ9Yu6OK8eV', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 185, 'out

In [67]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "엔비디아 사도 돼?")
])

chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='엔비디아 사도 돼?', additional_kwargs={}, response_metadata={}, id='651c6c4b-5199-4ad8-a85b-32bd22eb45f7'),
  AIMessage(content='엔비디아(NVIDIA)는 인공지능(AI), 그래픽 카드, 데이터 센터 등 다양한 분야에서 활동하는 기술 기업입니다. 주식 투자를 고려할 때는 여러 가지 요소를 신중하게 분석해야 합니다. \n\n1. **재무 성과**: 최근의 수익 보고서와 재무 상태를 검토하세요.\n2. **시장 동향**: AI와 같은 성장 산업의 동향을 살펴보세요.\n3. **경쟁사 분석**: AMD, 인텔 등 경쟁사의 동향도 중요합니다.\n4. **전문가 의견**: 애널리스트의 평가 및 가격 목표를 참고하세요.\n5. **포트폴리오 다각화**: 투자 포트폴리오의 균형을 고려하세요.\n\n전문가의 조언이나 충분한 시장 조사가 필요합니다. 또한, 본인의 투자 습관과 리스크 수용 능력도 고려해야 합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 203, 'prompt_tokens': 183, 'total_tokens': 386, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_22a0c4db5d', 'id': '

# Stock Agent
1. DuckDuckgo search tool
	- 회사 정보를 웹에서 찾는 툴을 만들기
	- 회사가 상장했는가, 회사 주식 심볼(ticker, 티커)는 무엇인지?
	- 가령 A라는 회사에 대한 정보를 찾고자 한다면, agent에 의해 툴은 A가 어떤 회사인지 검색을 수행하게 한다

2. AlphaVantaga API(주식 회사 정보)
	- 1) 회사의 심볼을 알아내는 툴
	- 2) 손익 계산서를 위한 툴
	- 3) 뉴스 심리지수를 위한 툴
	- 4) 회사의 개요를 위한 툴

	- https://www.alphavantage.co/
	- 위 사이트에 접속 후 API_KEY 발급
	- 환경변수에 등록하기
	- ALPHA_VANTAGE_API_KEY="발급받은 키"

In [2]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List, Tuple, Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-VZytJ5BN3ErM


## 1. 일, 주, 월 단위 실적을 제공 
- https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey=demo

In [2]:
"""
    {
    "Meta Data": {
        "1. Information": "Daily Prices (open, high, low, close) and Volumes",
        "2. Symbol": "IBM", #회사의 티커
        "3. Last Refreshed": "2026-05-18",
        "4. Output Size": "Compact",
        "5. Time Zone": "US/Eastern"
    },
    "Time Series (Daily)": { 
        "2026-05-18": {
            "1. open": "218.5500", #시작 가격
            "2. high": "223.3300", #가장 최고가
            "3. low": "217.7500", #가장 최저가
            "4. close": "222.7500", #마감가
            "5. volume": "5946367" #거래량
        },
        "2026-05-15": {
            "1. open": "218.2000",
            "2. high": "220.9100",
            "3. low": "217.6150",
            "4. close": "219.3000",
            "5. volume": "6154450"
        },
"""

None

## 2. News & Sentiments (최신뉴스 & 민감도)
- https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey=demo

In [3]:
"""
    {
    "items": "50",
    # 뉴스의 지표
    "sentiment_score_definition": "x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish",
    "relevance_score_definition": "0 < x <= 1, with a higher score indicating higher relevance.",
    "feed": [
        {
            "title": "Apple CEO Tim Cook in Beijing with US Presidential Delegation – May 2026 - News and Statistics",
            "url": "https://www.indexbox.io/blog/tim-cook-joins-us-delegation-to-beijing-as-apple-navigates-china-ties/",
            "time_published": "20260519T031958",
            "authors": [],
            "summary": "Apple CEO Tim Cook is in Beijing as part of a U.S. presidential delegation, marking the first such visit in nearly a decade. This trip is crucial for Apple due to its extensive manufacturing operations in China and China being its largest market outside the U.S. Improved U.S.-China relations, including reduced tariffs and increased market access, would significantly benefit Apple, especially following Chinese President Xi Jinping's recent pledge to \"open wider\" for American businesses.",
            "banner_image": "https://www.indexbox.io/landing/img/blog/telegram-fallback/5eab958188a1a3d707e92b07af013cb6.webp",
            "source": "IndexBox",
            "category_within_source": "General",
            "source_domain": "IndexBox",
            "topics": [
                {
                    "topic": "technology",
                    "relevance_score": "0.812212"
                },
                {
                    "topic": "economy_macro",
                    "relevance_score": "0.745762"
                },
                {
                    "topic": "finance",
                    "relevance_score": "0.634877"
                },
                {
                    "topic": "manufacturing",
                    "relevance_score": "0.647957"
                }
            ],
            # 0.335609: 뉴스 지표 점수(Somewhat_Bullish)
            "overall_sentiment_score": 0.335609,
            "overall_sentiment_label": "Somewhat-Bullish",
            "ticker_sentiment": [
                {
                    "ticker": "AAPL",
                    "relevance_score": "1.000000",
                    "ticker_sentiment_score": "0.423983",
                    "ticker_sentiment_label": "Bullish"
                },
                {
                    "ticker": "NVDA",
                    "relevance_score": "0.610471",
                    "ticker_sentiment_score": "0.315554",
                    "ticker_sentiment_label": "Somewhat-Bullish"
                },
                {
                    "ticker": "TSLA",
                    "relevance_score": "0.611822",
                    "ticker_sentiment_score": "0.327856",
                    "ticker_sentiment_label": "Somewhat-Bullish"
                }
            ]
        },
"""

None

## 3. 회사 재무 재표, 손익(Income Statement)
- https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol=IBM&apikey=demo

In [5]:
"""
    {
    "symbol": "IBM",
    "annualReports": [
        {
            "fiscalDateEnding": "2025-12-31",
            "reportedCurrency": "USD",
            "grossProfit": "40185000000", ※순수익
            "totalRevenue": "67535000000", ※총매출
            "costOfRevenue": "27350000000", ※원가
            "costofGoodsAndServicesSold": "27350000000",
            "operatingIncome": "10325000000",
            "sellingGeneralAndAdministrative": "18285000000",
            "researchAndDevelopment": "8320000000",
            "operatingExpenses": "29860000000",
            "investmentIncomeNet": "None",
            "netInterestIncome": "-1290000000",
            "interestIncome": "645000000",
            "interestExpense": "1935000000",
            "nonInterestIncome": "None",
            "otherNonOperatingIncome": "None",
            "depreciation": "None",
            "depreciationAndAmortization": "5021000000",
            "incomeBeforeTax": "10328000000",
            "incomeTaxExpense": "-242000000",
            "interestAndDebtExpense": "None",
            "netIncomeFromContinuingOperations": "10571000000",
            "comprehensiveIncomeNetOfTax": "None",
            "ebit": "12263000000",
            "ebitda": "17284000000",
            "netIncome": "10593000000"
        },

"""

None

## 4. 회사의 개요(Company OverView)
- https://www.alphavantage.co/query?function=OVERVIEW&symbol=IBM&apikey=demo

## Stock Tools 생성

In [4]:
class StockMartSymbolSearchToolArgSchema(BaseModel):
    query: str = Field(description="""
        The query you will search for Example query: Stock Market Symbol for Apple company.
    """)

class StockMartSymbolSearchTool(BaseTool):
    name: Type[str] = "stock_mark_symbol_search_tool"
    description: Type[str] = """
        Use this tool fin the stock market symbol for a company.
        It takes a query as an argument.
    """

    args_schema: Type[BaseModel] = StockMartSymbolSearchToolArgSchema
    
    def _run(self, query):
        ddg = DuckDuckGoSearchAPIWrapper()
        ddg.run(query)

In [11]:
def parse_output(result):
    return result["messages"][-1].content

In [26]:
tools = [StockMartSymbolSearchTool()]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question": "엔비디아의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

In [30]:
print(result)

엔비디아(NVIDIA)의 주식 심볼을 찾지 못했습니다. 하지만 일반적으로 엔비디아의 주식 심볼은 "NVDA"입니다. 이 정보를 바탕으로 엔비디아에 대한 개요, 손익 계산서 및 뉴스 등을 확인해야 합니다.

1. **회사 개요**: 
   엔비디아는 그래픽 처리 장치(GPU)를 설계 및 제조하는 미국의 기술 기업입니다. 주로 비디오 게임, 데이터 센터, 인공지능 및 머신러닝 분야에서 강력한 솔루션을 제공합니다.

2. **손익 계산서**:
   - 매출: 엔비디아는 최근 몇 년간 매출 성장이 두드러졌습니다.
   - 순이익: 높은 마진을 자랑하며 지속적인 수익성을 유지하고 있습니다.

3. **뉴스**:
   최근 엔비디아는 AI 및 클라우드 컴퓨팅 분야에서 강력한 성과를 보이며 주가가 상승세를 보이고 있습니다. 새로운 제품 출시와 함께 시장 점유율 확대에 대한 기대감이 커지고 있습니다.

이제 엔비디아에 투자할지 여부를 판단하기 위해 고려할 사항들은 다음과 같습니다:
- 현재 주가가 앞으로의 성장 가능성과 비즈니스 전망에 비해 적정한지 여부
- 엔비디아와 경쟁하는 다른 기업들과의 비교
- 경제 전반의 상황 및 기술 산업의 동향
- 개인의 투자 목표 및 리스크 수용 능력

위 정보를 종합해 보아 엔비디아는 앞으로도 성장 가능성이 크지만, 투자 결정을 내리기 전 항상 추가적인 정보와 분석을 참고하는 것이 중요합니다.


In [5]:
alpha_vantage_api_kay = os.environ.get("ALPHA_VANTAGE_API_KEY")
alpha_vantage_api_kay

'4BXQCJFYGJL5JTQ3'

## News Sentiments Tool

In [6]:
class NewsSentimentToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Example: AAPL, TSLA")

class NewsSentimentTool(BaseTool):
    name: Type[str] = "news_sentiment_tool"
    description: Type[str] = """
        Use this to get the news sentiment, recent trends of a company.
        You should enter a stock symbol.
    """

    args_schema: Type[BaseModel] = NewsSentimentToolArgsSchema

    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={symbol}&apikey={alpha_vantage_api_kay}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [35]:
tools = [
    StockMartSymbolSearchTool(), # 회사 심볼 툴
    NewsSentimentTool(), # 회사 최근 뉴스, 민감도 툴
        ]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question": "엔비디아의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

In [36]:
print(result)

### 엔비디아(NVIDIA Corporation) 개요

**심볼:** NVDA

**회사 개요:**
엔비디아(NVIDIA Corporation)는 그래픽 처리 장치(GPU) 및 AI(인공지능) 솔루션을 설계하고 개발하는 미국의 기술 회사입니다. 1993년 설립된 엔비디아는 주로 컴퓨터 그래픽, 모바일 컴퓨팅, 그리고 AI 기계 학습의 발전에 기여하는 GPU 및 SoC(시스템 온 칩) 기술로 유명합니다. 최근에는 AI 및 자율 주행차 기술에 집중하고 있으며, 데이터 센터 및 클라우드 컴퓨팅에 필요한 고성능 컴퓨팅 솔루션을 제공하고 있습니다.

### 최근 뉴스 및 투자 심리

**뉴스 요약:**
1. **중국 시장 접근:** 엔비디아 CEO는 중국이 향후 미국의 AI 칩을 수용할 가능성이 높다고 언급했습니다. 이는 중국과의 관계 개선에 대한 기대를 불러일으킵니다. [출처](https://www.bloomberg.com/news/articles/2026-05-18/nvidia-s-ceo-says-china-will-open-its-market-to-ai-chips-from-us)
   
2. **AI 인프라 회사 성장:** 엔비디아가 지원하는 AI 인프라 회사가 데이터 센터 용량을 급속히 확장하고 있으며, AI 수요 증가에 따라 성장 가능성이 높은 상태입니다. [출처](https://www.theglobeandmail.com/investing/markets/stocks/GS-N/pressreleases/2003205/)
   
3. **시장의 전반적인 짠 전망:** 그러나 일부 기관 투자자들은 엔비디아 주식에 대한 공매도를 시작했으며, 이는 주식에 대한 우려를 나타내고 있습니다. [출처](https://www.benzinga.com/markets/prediction-markets/26/05/52638816/why-ai-wunderkind-leopold-aschenbrenner-started-betting-against-nvidia-oracle-broadcom)

## 회사의 재무재표, 손익 툴(income statement)

In [7]:
class CompanyIncomeStatementToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyIncomeStatementTool(BaseTool):
    name: Type[str] = "company_income_statement_tool"
    description: Type[str] = """
        Use this to get the income statement of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyIncomeStatementToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={alpha_vantage_api_kay}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [38]:
tools = [
    StockMartSymbolSearchTool(), # 회사 심볼 툴
    NewsSentimentTool(), # 회사 최근 뉴스, 민감도 툴
    CompanyIncomeStatementTool(), #회사의 재무재표 손익 툴
        ]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question": "엔비디아의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

In [39]:
print(result)

### 엔비디아(NVIDIA Corporation)에 대한 종합 분석

#### 1. 심볼
- 엔비디아의 주식 심볼은 **NVDA**입니다.

#### 2. 회사 개요
엔비디아는 그래픽 처리 장치(GPU) 및 인공지능(AI) 기술을 전문으로 하는 미국의 기술 기업으로, 주로 비디오 게임, 데이터 센터, 전문 비주얼 컴퓨팅, 인공지능 시스템 및 자율주행 자동차 분야에 집중하고 있습니다. 최근 몇 년간 인공지능에 대한 수요가 급증하면서 엔비디아의 기술과 제품은 더욱 주목받고 있습니다.

#### 3. 손익 계산서
(2026년 1월 31일 종료 기준, 단위: USD)
- 총 수익: 215,938,000,000
- 총 매출 원가: 62,475,000,000
- 총 매출 총 이익: 153,463,000,000
- 운영 수익: 130,387,000,000
- 연구 개발 비용: 18,497,000,000
- 순이익: 120,067,000,000

#### 4. 뉴스 sentiment 및 최근 경향
- 최근 뉴스 sentiment는 **중립**에서 **다소 긍정적**으로 평가받고 있으며, AI 및 반도체 분야에서의 성장 가능성에 대한 긍정적인 시각이 있습니다. 
- 엔비디아 CEO는 중국 시장에서 AI 칩의 수요가 증가할 것으로 예상하고 있으며, 이러한 전망은 주가에 긍정적인 영향을 미칠 것으로 예상됩니다.
  
#### 5. 결론: 엔비디아 구매 여부
- **장점:**
  - 강력한 재무 성과 (높은 수익성과 순이익)
  - AI 및 데이터 센터 분야의 성장 잠재력
  - 최근 긍정적인 시장 및 기술 트렌드

- **단점:**
  - 경기 변동성에 따른 위험성
  - 경쟁 심화 (특히 AMD, Intel 등과의 경쟁)

결론적으로, 엔비디아는 기술 산업의 성장을 이끄는 주요 기업으로, 긍정적인 재무 성과와 시장 전망이 우세한 편입니다. 만약 기술 주식 및 AI 관련 기업에 대한 투자를 고려하고 있다면, 엔비디아는 좋은 선택이 될 수 있습니다. 다만, 시장의 변동성 및 경쟁

## 회사 개요 툴, 주가 정보 툴 

In [8]:
class CompanyOverviewToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyOverviewTool(BaseTool):
    name: Type[str] = "company_overview_Tool"
    description: Type[str] = """
        Use this to get the overview of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyOverviewToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_vantage_api_kay}"
        response = requests.get(url)
        datas = response.json()
        return datas


class CompanyStockPerformanceToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyStockPerformanceTool(BaseTool):
    name: Type[str] = "company_stock_performance_tool"
    description: Type[str] = """
        Use this to get the weekly performance of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyStockPerformanceToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol={symbol}&apikey={alpha_vantage_api_kay}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [9]:
tools = [
    StockMartSymbolSearchTool(), # 회사 심볼 툴
    NewsSentimentTool(), # 회사 최근 뉴스, 민감도 툴
    CompanyIncomeStatementTool(), #회사의 재무재표 손익 툴
    CompanyOverviewTool(), #회사 개요 툴
    CompanyStockPerformanceTool(), # 한 주간의 주가 정보를 알아오는 툴
]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a veteran Wall Street stock investment expert and a cold-blooded Chief Financial Analyst. 
        Your task is to comprehensively analyze the company's financial overview, income statement, and recent stock price trends to provide sharp, data-driven investment insights.

        [CORE DIRECTIVES]
        1. Avoid ambiguity. Never provide vague or irresponsible answers like "it depends on the investor's choice" or "it is difficult to predict."
        2. Make a definitive call. Based on the retrieved data, you MUST provide a clear and explicit final investment conclusion: choosing exactly one from [BUY / HOLD / SELL].
        3. Maintain a highly professional, objective, and authoritative tone. Back up your conclusion logically using concrete numbers and financial metrics (profitability, growth, and price momentum).
        4. Language Requirement: You MUST write the final answer entirely in Korean. Even though the analysis is based on English data, the final output delivered to the user must be in clear, professional Korean.
        
        Your analysis will guide critical financial decisions. Be ruthless, objective, and strictly rely on the data provided.
    """),
    ("human", """
        You must use tools to answer this question.
    
        1. Find the stock symbol for {company}.
        2. Retrieve the company's financial overview.
        3. Retrieve the company's income statement.
        4. Retrieve the stock price data (recent price, trend, or performance).
        5. Retrieve at least 5 recent news articles for {company} along with their sources (publisher or URL), and analyze the overall news sentiment.
        6. Based on ALL of the following:
        - Financial data
        - Income statement
        - Stock price performance
    
        Analyze whether {company} is a good investment.
    
        Final answer must include:
        - Stock symbol
        - Key financial metrics
        - Income insights (revenue, net income)
        - Stock price trend
        - Investment conclusion
    """),
])

chain = prompt | agent | RunnableLambda(parse_output)

In [13]:
result = chain.invoke({
   "company": "엔비디아" 
})

In [14]:
print(result)

현재 사용할 수 있는 API 리소스의 제한으로 인해 엔비디아(NVIDIA)와 관련된 정보를 수집할 수 없습니다. 그러나 일반적으로 엔비디아는 반도체 산업에서 매우 강력한 위치를 차지하고 있으며, GPU와 AI 칩의 수요가 급증함에 따라 재무 성과가 강력하게 유지되고 있습니다.

추가적인 재무 데이터를 수집하는 것은 불가능하지만, 엔비디아의 최근 성과는 다음과 같은 트렌드들이 관찰되었습니다:

1. **재무 성과:** 엔비디아의 매출은 최근 몇 년 동안 급격히 증가했습니다. AI 및 게임용 GPU의 수요가 이를 이끌었고, 이는 순이익 증가로 이어졌습니다.

2. **주가 성과:** 엔비디아의 주가는 최근 몇 년 동안 강력한 상승세를 보였으며, 이는 클라우드 컴퓨팅과 AI 기술에 대한 수요 증가의 반영입니다.

3. **뉴스 감정:** 업계 전문가들의 분석 및 뉴스는 대체로 긍정적인 편이며, 이는 엔비디아의 새로운 제품 라인업과 협업이 긍정적으로 작용하고 있기 때문입니다.

결론적으로, 엔비디아는 강력한 재무성과와 긍정적인 시장 감정으로 인해 **BUY** 결정을 내리는 것이 합리적입니다. 투자자들은 이 회사를 장기적으로 보유하는 것을 고려해야 합니다.
